# NutriEpiDB: A Database For Dietary Compounds With Epigenetic Targets Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the NutriEpiDB dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.sx3s-9110/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("\nLoaded Dataset Metadata:")
print("Name:", metadata['name'])
print("Description:", metadata['description'])
print("Published:", metadata.get('datePublished', 'N/A'))
print("License:", metadata.get('license', 'N/A'))


## 2. Data Overview
Review available record sets, fields, and their IDs. This gives an overview of the dataset structure, including the entities defined in the Croissant schema.

In [ ]:
# List available record sets and their fields
print("\nAvailable Record Sets:")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print("- RecordSet Name:", getattr(rs, 'name', 'N/A'))
    print("  @id:", rs.id)
    print("  Description:", getattr(rs, 'description', ''))
    print("  Available Fields and Columns:")
    fields = getattr(rs, 'fields', [])
    for f in fields:
        print("    * Field Name:", getattr(f, 'name', 'N/A'))
        print("      @id:", f.id)
        print("      DataType:", getattr(f, 'data_type', ''))
        columns = getattr(f, 'columns', [])
        for c in columns:
            print("      - Column Name:", getattr(c, 'name', 'N/A'))
            print("        @id:", c.id)
            print("        DataType:", getattr(c, 'data_type', ''))
    print("")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note:** For this dataset, we'll extract available record sets using their `@id` values. If multiple record sets exist, we loop through all.

In [ ]:
# Extract records from each record set
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print("Columns:", df.columns.tolist())
        print(df.head(2))
    else:
        print("No records found for this record set.")

# For demonstration, select the first non-empty record set
sample_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        sample_record_set_id = rid
        break

if sample_record_set_id:
    print(f"\nSample RecordSet chosen for analysis: {sample_record_set_id}")
    print("Columns:", dataframes[sample_record_set_id].columns.tolist())
    display(dataframes[sample_record_set_id].head())
else:
    print("No dataframes with records found.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

If numeric columns are available, demonstrate normalization and grouping.

In [ ]:
# Find a numeric field in the selected record set
import numpy as np
if sample_record_set_id:
    df = dataframes[sample_record_set_id]

    # Try to detect numeric columns
    numeric_columns = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    groupable_columns = [col for col in df.columns if df[col].dtype == object and df[col].nunique() < min(30, int(0.1 * len(df)))]
    print("\nNumeric columns candidates:", numeric_columns)
    print("Groupable columns candidates:", groupable_columns)

    # Example operation: use first numeric and groupable fields found
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        threshold = df[numeric_field_id].mean()  # Example threshold: mean value
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by groupable field
        if groupable_columns:
            group_field_id = groupable_columns[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric columns found for EDA.")
else:
    print("No sample record set available for EDA.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll plot a histogram and a grouped bar plot if EDA yielded results.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if sample_record_set_id and numeric_columns:
    df = dataframes[sample_record_set_id]

    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Grouped bar plot
    if groupable_columns and numeric_field_id:
        group_field_id = groupable_columns[0]
        grouped_df = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8, 4))
        sns.barplot(x=grouped_df[group_field_id], y=grouped_df[numeric_field_id])
        plt.xticks(rotation=45)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The NutriEpiDB Croissant schema describes a rich dataset of dietary compounds and their epigenetic targets.
- By referencing entities by their `@id` fields, we ensured traceability and precise linkage to schema components.
- Basic EDA and visualization enabled preliminary insight into quantitative attributes such as binding affinity, epigenetic effects, or compound properties.
- The approach illustrated here can be extended to more advanced analytics or machine learning workflows using this standardized, FAIR dataset.